# 8.1 distillations for reasoning model
- What is distillation it is about your train smaller LM, which means the students on the output produced by Elijah LM more intelligent how am we sensei the teacher.
- two kind of distillation heart and soft heart is about matching the token so is about matching the logits.


In [ ]:
from reasoning_from_scratch.ch08 import load_reasoning_tokenizer

tokenizer = load_reasoning_tokenizer()

tokenizer-reasoning.json: 100% (10 MiB / 10 MiB)


In [ ]:
prompt = "Sam is hired for a 20-day period..."
prompt_ids = tokenizer.encode(prompt, chat_wrapped=True)
decoded_prompt = tokenizer.decode(prompt_ids)
print(decoded_prompt)

<|im_start|>user
Sam is hired for a 20-day period...<|im_end|>
<|im_start|>assistant



In [ ]:
answer = "<think>Okay, let me try to solve this problem...</think> \\boxed{4}"
answer_ids = tokenizer.encode(answer, chat_wrapped=False)
decoded_answer = tokenizer.decode(answer_ids)
print(decoded_answer)


<think>Okay, let me try to solve this problem...</think> \boxed{4}


In [6]:
token_ids = prompt_ids + answer_ids + [tokenizer.eos_token_id]
decoded_token_ids = tokenizer.decode(token_ids)
print(decoded_token_ids)


<|im_start|>user
Sam is hired for a 20-day period...<|im_end|>
<|im_start|>assistant
<think>Okay, let me try to solve this problem...</think> \boxed{4}<|im_end|>


# 8.4 preparing dataset

In [ ]:
from reasoning_from_scratch.ch08 import build_examples, load_distill_data

math_train = load_distill_data(partition="deepseek-r1-math-train")

print("Dataset size:", len(math_train))

examples, skipped = build_examples(math_train, tokenizer)

skipped

deepseek-r1-math-train.json: 107538.0 KB
Dataset size: 12000


0

In [ ]:
from pprint import pprint

pprint(tokenizer.decode(examples[0]["token_ids"]))

('<|im_start|>user\n'
 'You are a helpful math assistant.\n'
 'Answer the question and write the final result on a new line as:\n'
 '\\boxed{ANSWER}\n'
 '\n'
 'Question:\n'
 'Let \\[f(x) = \\left\\{\n'
 '\\begin{array}{cl} ax+3, &\\text{ if }x>2, \\\\\n'
 'x-5 &\\text{ if } -2 \\le x \\le 2, \\\\\n'
 '2x-b &\\text{ if } x <-2.\n'
 '\\end{array}\n'
 '\\right.\\]Find $a+b$ if the piecewise function is continuous (which means '
 'that its graph can be drawn without lifting your pencil from the paper).\n'
 '\n'
 'Answer:<|im_end|>\n'
 '<|im_start|>assistant\n'
 "<think>Okay, so I need to find the values of 'a' and 'b' such that the "
 'piecewise function f(x) is continuous everywhere. Then add them together for '
 'the final answer. Let me recall that for a piecewise function to be '
 'continuous, the left-hand limit and right-hand limit at each boundary point '
 "must equal the function's value at that point. The boundaries here are at x "
 '= -2 and x = 2. Let me start by checking each b

In [ ]:
from reasoning_from_scratch.ch08 import filter_examples_by_max_len

filtered_examples = filter_examples_by_max_len(examples, max_len=2048)

import random

rng = random.Random(123)
rng.shuffle(filtered_examples)

train_examples = filtered_examples[25:]
val_examples = filtered_examples[:25]

Original: 12000
Filtered: 6695
Removed: 5305


In [ ]:
# 8.5 load of pre-trained model
import torch

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_model_and_tokenizer,
)

device = get_device()

model, _ = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
)

Using Apple Silicon GPU (MPS)
qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


In [21]:
# 5730 is the shortest example in train_examples
token_ids = train_examples[5730]["token_ids"]
prompt_len = train_examples[5730]["prompt_len"]

from reasoning_from_scratch.ch06 import sequence_logprob

tok = torch.tensor(token_ids, dtype=torch.long, device=device)

with torch.no_grad():
    seq_logprob = sequence_logprob(model, tok, prompt_len)
    num_answer_tokens = tok.numel() - prompt_len
    avg_logprob = -seq_logprob / num_answer_tokens
print(f"Average logprob: {avg_logprob:.2f}")

Average logprob: 1.68


In [ ]:
# computing the training and evaluation loss
input_ids = tok[:-1].unsqueeze(0)
target_ids = tok[1:]  # Targets are inputs shifted by one position
logits = model(input_ids).squeeze(0)

# Remove prompt tokens
first_answer_logit_idx = max(prompt_len - 1, 0)
answer_logits = logits[first_answer_logit_idx:]
answer_targets = target_ids[first_answer_logit_idx:]

with torch.no_grad():
    ce_mean_direct = torch.nn.functional.cross_entropy(answer_logits, answer_targets)
print(f"Cross-entropy: {ce_mean_direct:.2f}")

Cross-entropy: 1.68


# what i learned:
- prepare dataset
- prepare loss function
- prepare training loop
- evaluate and visualize

In [ ]:
def save_checkpoint(model, checkpoint_dir, step, suffix=""):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    suffix = f"-{suffix}" if suffix else ""
    ckpt_path = checkpoint_dir / f"qwen3-0.6B-distill-step{step:05d}{suffix}.pth"
    torch.save(model.state_dict(), ckpt_path)
    return ckpt_path


def append_csv_metrics(
    csv_log_path,
    epoch_idx,
    total_steps,
    train_loss,
    val_loss,
):
    if not csv_log_path.exists():
        csv_log_path.write_text(
            "epoch,total_steps,train_loss,val_loss\n",
            encoding="utf-8",
        )
    with csv_log_path.open("a", encoding="utf-8") as f:
        f.write(f"{epoch_idx},{total_steps},{train_loss:.6f},{val_loss:.6f}\n")

In [ ]:
import time
from reasoning_from_scratch.ch08 import compute_example_loss, evaluate_examples


def train_distillation(
    model,
    train_examples,
    val_examples,
    device,
    epochs=2,
    lr=5e-6,
    grad_clip_norm=None,
    seed=123,
    log_every=50,
    checkpoint_dir="checkpoints",
    csv_log_path=None,
):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()

    total_steps = epochs * len(train_examples)
    global_step = 0
    rng = random.Random(seed)

    if csv_log_path is None:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        csv_log_path = f"train_distill_metrics_{timestamp}.csv"
    csv_log_path = Path(csv_log_path)

    for epoch in range(1, epochs + 1):
        epoch_examples = list(train_examples)
        rng.shuffle(epoch_examples)

        for example in epoch_examples:
            global_step += 1

            optimizer.zero_grad()

            loss = compute_example_loss(model, example, device)

            loss.backward()

            # Optionally clip large gradients to improve training stability
            if grad_clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
            # Step 8: update the model weights
            optimizer.step()

            # Step 9: periodically evaluate the current model on the validation set
            if log_every and global_step % log_every == 0:
                val_loss = evaluate_examples(
                    model=model,
                    examples=val_examples,
                    device=device,
                )

                print(
                    f"[Epoch {epoch}/{epochs} "
                    f"Step {global_step}/{total_steps}] "
                    f"train_loss={loss.item():.4f} "
                    f"val_loss={val_loss:.4f}"
                )
                append_csv_metrics(
                    csv_log_path=csv_log_path,
                    epoch_idx=epoch,
                    total_steps=global_step,
                    train_loss=loss.item(),
                    val_loss=val_loss,
                )

        model.train()
        # Step 10: save a checkpoint for this epoch
        ckpt_path = save_checkpoint(
            model=model,
            checkpoint_dir=checkpoint_dir,
            step=global_step,
            suffix=f"epoch{epoch}",
        )
        print(f"Saved checkpoint to {ckpt_path}")
    return model

In [ ]:
torch.manual_seed(0)

train_distillation(
    model,
    train_examples=train_examples[:10],
    val_examples=val_examples[:10],
    device=device,
    epochs=2,
    lr=5e-6,
    grad_clip_norm=1.0,  # Same as in chapter 6
    seed=123,
    log_every=5,
    csv_log_path="train_distill_metrics.csv",
)